# Задача 9: Метод Зейделя (Gauss-Seidel)
---

**Задача:** Реализовать метод Зейделя для решения СЛАУ на языке Python с использованием циклов и условных операторов.

### Теория

**Метод Зейделя** — итерационный метод решения СЛАУ $Ax = b$.

На каждой итерации $k$ новое приближение вычисляется по формуле:

$$x_i^{(k)} = \frac{1}{a_{ii}} \left( b_i - \sum_{j=1}^{i-1} a_{ij} x_j^{(k)} - \sum_{j=i+1}^{n} a_{ij} x_j^{(k-1)} \right), \quad i = 1, \ldots, n$$

**Ключевое отличие от метода Якоби:**
- В методе Якоби используются только значения **предыдущей** итерации: $x_j^{(k-1)}$
- В методе Зейделя используются **уже обновлённые** значения текущей итерации: $x_j^{(k)}$ для $j < i$

**Условие сходимости:** Достаточное условие — строгое диагональное преобладание:

$$|a_{ii}| > \sum_{j=1, j \neq i}^{n} |a_{ij}|, \quad \forall i$$

In [ ]:
import numpy as np

### Проверка диагонального преобладания

In [ ]:
def check_diagonal_dominance(A):
    """
    Проверка условия строгого диагонального преобладания.
    
    Условие: |a_ii| > sum_{j≠i} |a_ij| для всех i
    
    Параметры:
    A : матрица системы (список списков)
    
    Возвращает:
    True, если условие выполнено, иначе False
    """
    n = len(A)
    for i in range(n):
        diagonal = abs(A[i][i])
        sum_off_diagonal = sum(abs(A[i][j]) for j in range(n) if j != i)
        if diagonal <= sum_off_diagonal:
            return False
    return True

### Реализация метода Зейделя

In [ ]:
def gauss_seidel(A, b, eps=1e-6, max_iter=1000):
    """
    Метод Зейделя для решения СЛАУ Ax = b.
    
    Формула:
    x_i^(k) = (1/a_ii) * [b_i - sum_{j<i} a_ij*x_j^(k) - sum_{j>i} a_ij*x_j^(k-1)]
    
    Параметры:
    A : матрица системы (список списков)
    b : вектор правой части (список)
    eps : точность (критерий остановки)
    max_iter : максимальное число итераций
    
    Возвращает:
    x : вектор решения
    k : число итераций
    """
    n = len(A)
    
    # Проверка диагонального преобладания
    if not check_diagonal_dominance(A):
        print("Предупреждение: нет диагонального преобладания, сходимость не гарантирована")
    
    # Начальное приближение: x^(0) = (0, 0, ..., 0)
    x = [0.0] * n
    
    for k in range(1, max_iter + 1):
        x_old = x.copy()  # Сохраняем предыдущее приближение для проверки сходимости
        
        # Обновляем каждую компоненту
        for i in range(n):
            sum_val = b[i]
            
            # Вычитаем сумму с уже обновлёнными значениями (j < i)
            for j in range(i):
                sum_val -= A[i][j] * x[j]  # используем x[j] - уже обновлённое значение
            
            # Вычитаем сумму со старыми значениями (j > i)
            for j in range(i + 1, n):
                sum_val -= A[i][j] * x_old[j]  # используем x_old[j] - старое значение
            
            # Обновляем x[i]
            x[i] = sum_val / A[i][i]
        
        # Проверка критерия остановки: ||x^(k) - x^(k-1)||_inf < eps
        diff = max(abs(x[i] - x_old[i]) for i in range(n))
        if diff < eps:
            return x, k
    
    return x, max_iter

### Вспомогательные функции

In [ ]:
def print_vector(name, v):
    """Красивый вывод вектора"""
    print(f"{name} = [{', '.join(f'{vi:.6f}' for vi in v)}]\n")

def mat_vec(A, x):
    """Умножение матрицы на вектор: Ax"""
    return [sum(A[i][j] * x[j] for j in range(len(x))) for i in range(len(A))]

def compute_residual(A, x, b):
    """Вычисление невязки ||Ax - b||_inf"""
    Ax = mat_vec(A, x)
    return max(abs(Ax[i] - b[i]) for i in range(len(b)))

---
## Тест 1: Пример из лекции (2×2)

$$A = \begin{pmatrix} 2 & 1 \\ 1 & 2 \end{pmatrix}, \quad b = \begin{pmatrix} 1 \\ -1 \end{pmatrix}$$

Точное решение: $x^* = (1, -1)$

In [ ]:
A2 = [
    [2, 1],
    [1, 2],
]
b2 = [1, -1]

x2, iters2 = gauss_seidel(A2, b2, eps=1e-6)

print("="*60)
print("  МЕТОД ЗЕЙДЕЛЯ — пример из лекции (2×2)")
print("="*60)
print(f"Диагональное преобладание: {check_diagonal_dominance(A2)}")
print(f"Итераций: {iters2}")
print()
print_vector("x (результат)", x2)
print_vector("x* (точное)  ", [1.0, -1.0])

residual2 = compute_residual(A2, x2, b2)
print(f"Невязка ||Ax - b||_inf = {residual2:.2e}")

---
## Тест 2: Матрица 4×4 с диагональным преобладанием

$$A = \begin{pmatrix} 10 & 1 & 1 & 1 \\ 1 & 10 & 1 & 1 \\ 1 & 1 & 10 & 1 \\ 1 & 1 & 1 & 10 \end{pmatrix}, \quad b = \begin{pmatrix} 13 \\ 13 \\ 13 \\ 13 \end{pmatrix}$$

Точное решение: $x^* = (1, 1, 1, 1)$

In [ ]:
A4 = [
    [10, 1, 1, 1],
    [1, 10, 1, 1],
    [1, 1, 10, 1],
    [1, 1, 1, 10],
]
b4 = [13, 13, 13, 13]

x4, iters4 = gauss_seidel(A4, b4, eps=1e-6)

print("="*60)
print("  МЕТОД ЗЕЙДЕЛЯ — матрица 4×4")
print("="*60)
print(f"Диагональное преобладание: {check_diagonal_dominance(A4)}")
print(f"Итераций: {iters4}")
print()
print_vector("x (результат)", x4)
print_vector("x* (точное)  ", [1.0, 1.0, 1.0, 1.0])

residual4 = compute_residual(A4, x4, b4)
print(f"Невязка ||Ax - b||_inf = {residual4:.2e}")

---
## Тест 3: Матрица 3×3

$$A = \begin{pmatrix} 4 & 1 & 0 \\ 1 & 4 & 1 \\ 0 & 1 & 4 \end{pmatrix}, \quad b = \begin{pmatrix} 5 \\ 6 \\ 5 \end{pmatrix}$$

In [ ]:
A3 = [
    [4, 1, 0],
    [1, 4, 1],
    [0, 1, 4],
]
b3 = [5, 6, 5]

x3, iters3 = gauss_seidel(A3, b3, eps=1e-6)

print("="*60)
print("  МЕТОД ЗЕЙДЕЛЯ — матрица 3×3")
print("="*60)
print(f"Диагональное преобладание: {check_diagonal_dominance(A3)}")
print(f"Итераций: {iters3}")
print()
print_vector("x (результат)", x3)

residual3 = compute_residual(A3, x3, b3)
print(f"Невязка ||Ax - b||_inf = {residual3:.2e}")

---
## Проверка через numpy

In [ ]:
print("="*60)
print("  СРАВНЕНИЕ С NUMPY")
print("="*60)

for A, b, x, name in [(A2, b2, x2, "2×2"), (A3, b3, x3, "3×3"), (A4, b4, x4, "4×4")]:
    x_np = np.linalg.solve(np.array(A, dtype=float), np.array(b, dtype=float))
    diff = max(abs(x[i] - x_np[i]) for i in range(len(x)))
    print(f"{name}: макс. расхождение с numpy = {diff:.2e}")

---
## Сравнение с методом Якоби

In [ ]:
def jacobi(A, b, eps=1e-6, max_iter=1000):
    """
    Метод Якоби для сравнения.
    Использует только значения предыдущей итерации.
    """
    n = len(A)
    x = [0.0] * n
    
    for k in range(1, max_iter + 1):
        x_new = [(b[i] - sum(A[i][j] * x[j] for j in range(n) if j != i)) / A[i][i] for i in range(n)]
        
        diff = max(abs(x_new[i] - x[i]) for i in range(n))
        x = x_new
        
        if diff < eps:
            return x, k
    
    return x, max_iter

In [ ]:
print("="*60)
print("  СРАВНЕНИЕ: ЗЕЙДЕЛЬ vs ЯКОБИ")
print("="*60)
print(f"{'Система':<10} {'Зейдель (итер.)':<20} {'Якоби (итер.)':<20} {'Ускорение'}")
print("-"*60)

for A, b, name in [(A2, b2, "2×2"), (A3, b3, "3×3"), (A4, b4, "4×4")]:
    _, iters_gs = gauss_seidel(A, b, eps=1e-6)
    _, iters_j = jacobi(A, b, eps=1e-6)
    speedup = iters_j / iters_gs
    print(f"{name:<10} {iters_gs:<20} {iters_j:<20} {speedup:.2f}x")

print("-"*60)
print("\nВывод: Метод Зейделя обычно сходится быстрее метода Якоби,")
print("так как использует уже обновлённые значения на текущей итерации.")